# Image Captioning

## CSCI E-25     

## Introduction       

In this notebook you will work with a **multi-modal variational autoencoder (VaE)** trained to generate image captions. The model contains three major blocks.
1. **Image encoder:** A CNN is used to compute an image embedding. In this example, we use a pretrained EfficientNetB0 model.       
2. **Text encoder:** A transformer model for computing the embeddings of the text.
3. **Decoder:** A transformer model using cross-attention between the embedded image and embedded text to generate text caption strings.    

This model operates in two modes, training model and inference mode.   
1. During training, images and captions are used to learn the weights for the model the minimize the loss between generated captions and the provided captions. In many cases, these blocks are trained simultaneously using a categorical cross entropy loss function. In this case, the weights of the pretrained model are frozen, and only text encoder and decoder are trained.
2. At inference time, the images are encoded and text captions are generated.    

> Attribution: The code in this notebook is is based on a [Keras example notebook](https://keras.io/examples/vision/image_captioning/) originally created by [A_K_Nain](https://twitter.com/A_K_Nain), 2021/05/29, updated 2021/10/31.   

## Setup to Run the Notebook  

This notebook was constructed and testing in Google Colab Pro+. You are welcome to use any environment of your choosing that will execute correctly.  

Execute the code in the cell below to import the required packages.   

In [ ]:
import os

os.environ["KERAS_BACKEND"] = "tensorflow"

import re
import numpy as np
import matplotlib.pyplot as plt

import tensorflow as tf
import keras
from keras import layers
from keras.applications import efficientnet
from keras.layers import TextVectorization

keras.utils.set_random_seed(111)

## Download the dataset

We will use the [Flickr8K dataset](https://github.com/jbrownlee/Datasets/blob/master/Flickr8k_Dataset.names) for this notebook. This dataset comprises over 8,000 images, paired with five different captions. Execute the code in the cell below to import this dataset.  

In [ ]:
# Download images and captions showing a progress bar
!wget -q --show-progress https://github.com/jbrownlee/Datasets/releases/download/Flickr8k/Flickr8k_Dataset.zip
!wget -q --show-progress https://github.com/jbrownlee/Datasets/releases/download/Flickr8k/Flickr8k_text.zip

# Unzip quietly without prompts removing existing files if required
!unzip -qq -o Flickr8k_Dataset.zip
!unzip -qq -o Flickr8k_text.zip

# Clean up zip files
!rm Flickr8k_Dataset.zip Flickr8k_text.zip

## Configure Hyperparameters

The code in the cell below sets some critical hyperparameters for the models in the notebook. Execute this code.  

In [ ]:
# Path to the images
IMAGES_PATH = "Flicker8k_Dataset"

# Desired image dimensions
IMAGE_SIZE = (299, 299)

# Vocabulary size
VOCAB_SIZE = 10000

# Fixed length allowed for any sequence
SEQ_LENGTH = 25

# Dimension for the image embeddings and token embeddings
EMBED_DIM = 512

# Per-layer units in the feed-forward network
FF_DIM = 512

# Other training parameters
BATCH_SIZE = 64
EPOCHS = 100
AUTOTUNE = tf.data.AUTOTUNE

## Preparing the Dataset

The code in the cell below prepares the data for training and validation of the image captioning model. The data structure created is then split into test and validation datasets. Execute this code.    

In [ ]:
def load_captions_data(filename):
    """Loads captions (text) data and maps them to corresponding images.

    Args:
        filename: Path to the text file containing caption data.

    Returns:
        caption_mapping: Dictionary mapping image names and the corresponding captions
        text_data: List containing all the available captions
    """

    with open(filename) as caption_file:
        caption_data = caption_file.readlines()
        caption_mapping = {}
        text_data = []
        images_to_skip = set()

        for line in caption_data:
            line = line.rstrip("\n")
            # Image name and captions are separated using a tab
            img_name, caption = line.split("\t")

            # Each image is repeated five times for the five different captions.
            # Each image name has a suffix `#(caption_number)`
            img_name = img_name.split("#")[0]
            img_name = os.path.join(IMAGES_PATH, img_name.strip())

            # We will remove caption that are either too short to too long
            tokens = caption.strip().split()

            if len(tokens) < 5 or len(tokens) > SEQ_LENGTH:
                images_to_skip.add(img_name)
                continue

            if img_name.endswith("jpg") and img_name not in images_to_skip:
                # We will add a start and an end token to each caption
                caption = "<start> " + caption.strip() + " <end>"
                text_data.append(caption)

                if img_name in caption_mapping:
                    caption_mapping[img_name].append(caption)
                else:
                    caption_mapping[img_name] = [caption]

        for img_name in images_to_skip:
            if img_name in caption_mapping:
                del caption_mapping[img_name]

        return caption_mapping, text_data


def train_val_split(caption_data, train_size=0.8, shuffle=True):
    """Split the captioning dataset into train and validation sets.

    Args:
        caption_data (dict): Dictionary containing the mapped caption data
        train_size (float): Fraction of all the full dataset to use as training data
        shuffle (bool): Whether to shuffle the dataset before splitting

    Returns:
        Traning and validation datasets as two separated dicts
    """

    # 1. Get the list of all image names
    all_images = list(caption_data.keys())

    # 2. Shuffle if necessary
    if shuffle:
        np.random.shuffle(all_images)

    # 3. Split into training and validation sets
    train_size = int(len(caption_data) * train_size)

    training_data = {
        img_name: caption_data[img_name] for img_name in all_images[:train_size]
    }
    validation_data = {
        img_name: caption_data[img_name] for img_name in all_images[train_size:]
    }

    # 4. Return the splits
    return training_data, validation_data


# Load the dataset
captions_mapping, text_data = load_captions_data("Flickr8k.token.txt")

# Split the dataset into training and validation sets
train_data, valid_data = train_val_split(captions_mapping)
print("Number of training samples: ", len(train_data))
print("Number of validation samples: ", len(valid_data))

Let's investigate the dictionary data structure created. To examine the first 5 key-value pairs in this data structure execute the code in the cell below.  

In [ ]:
N = 5
count = 0
for key, value in captions_mapping.items():
    print(f"Key: {key}, \n{value}")
    count += 1
    if count >= N:
        break

Notice that the key is a file path for an image and the value are the stings containing the captions. The caption strings begin with a <start> token and end with an <end> token.   

### Vectorizing the Text Data

Text must be numerically encoded for any computer algorithm to process it. The approach used here is fairly typical and involves two steps.    
1. The text string are vectorized by first tokenizing the string and then one hot encoding into a vector of integer values. The [Keras TextVectorization layer](https://keras.io/api/layers/preprocessing_layers/text/text_vectorization/) is used. The range of possible token values is set by the `VOCAB_SIZE` argument, with resulting vector of length `SEQ_LENGTH`. The documentation states the steps as:
   - Standardize the string by lowercasing + punctuation stripping.
   - Split each example into substrings, usually words.
   - Recombine substrings into tokens (usually ngrams).
   - Index tokens (associate a unique int value with each token).
   - Transform each example using this index, either into a vector of integers (in this example) or a dense float vector.
3. The vectorized text strings are embedded into a dense vector representation later in the processing.       

Execute the code to define a Keras layer to vectorize the text in the captions.  

In [ ]:
def custom_standardization(input_string):
    lowercase = tf.strings.lower(input_string)
    return tf.strings.regex_replace(lowercase, "[%s]" % re.escape(strip_chars), "")


strip_chars = "!\"#$%&'()*+,-./:;<=>?@[\]^_`{|}~"
strip_chars = strip_chars.replace("<", "")
strip_chars = strip_chars.replace(">", "")

vectorization = TextVectorization(
    max_tokens=VOCAB_SIZE,
    output_mode="int",
    output_sequence_length=SEQ_LENGTH,
    standardize=custom_standardization,
)
vectorization.adapt(text_data)

### Data Augmentation   

Data augmentation for the images is another required preprocessing layer. Execute the code in the cell below to define this layer.   

In [ ]:
# Data augmentation for image data
image_augmentation = keras.Sequential(
    [
        layers.RandomFlip("horizontal"),
        layers.RandomRotation(0.2),
        layers.RandomContrast(0.3),
    ]
)

## Building the Training Data Pipeline

We will now create a pipeline to generate pairs of images and corresponding captions for training and validation using a [tf.data.Dataset.from_tensor_slices](https://keras.io/guides/training_with_built_in_methods/) object. This pipeline uses the dictionaries created previously and executes two two steps.    
1. The image is read from the disk using the path that is the dictionary key.
2. The image is decoded, resized and put in TensorFlow `float32` format.
3. The data is batched for efficient processing.
4. The five captions corresponding to the image are tokenized and vectorized.

Execute this code to prepare the dataset batches.  

In [ ]:
def decode_and_resize(img_path):
    img = tf.io.read_file(img_path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, IMAGE_SIZE)
    img = tf.image.convert_image_dtype(img, tf.float32)
    return img


def process_input(img_path, captions):
    return decode_and_resize(img_path), vectorization(captions)


def make_dataset(images, captions):
    dataset = tf.data.Dataset.from_tensor_slices((images, captions))
    dataset = dataset.shuffle(BATCH_SIZE * 8)
    dataset = dataset.map(process_input, num_parallel_calls=AUTOTUNE)
    dataset = dataset.batch(BATCH_SIZE).prefetch(AUTOTUNE)

    return dataset


# Pass the list of images and the list of corresponding captions
train_dataset = make_dataset(list(train_data.keys()), list(train_data.values()))

valid_dataset = make_dataset(list(valid_data.keys()), list(valid_data.values()))

## Building the model

We are now ready to construct the image captioning model. The model comprises four components:

1. **A CNN:** used to extract the image features.   
2. **A TransformerEncoder:** Encodes the image features extracted by the CNN.
3. **A TransformerDecoder:** This model takes the encoder output and the embedded text data (sequences) as inputs and learns to generate captions.
4. **An ImageCaptioningModel:** This model contains the code train and evaluate the image captioning model.  

The first component is pretrained CNN model, `EfficientNetB0`, with the following key property.      
- The model weights are frozen.
- The output of the CNN is flattened into a 1-d vector.

Execute this code to load this function.  

In [ ]:
def get_cnn_model():
    base_model = efficientnet.EfficientNetB0(
        input_shape=(*IMAGE_SIZE, 3),
        include_top=False,
        weights="imagenet",
    )
    # We freeze our feature extractor
    base_model.trainable = False
    base_model_out = base_model.output
    base_model_out = layers.Reshape((-1, base_model_out.shape[-1]))(base_model_out)
    cnn_model = keras.models.Model(base_model.input, base_model_out)
    return cnn_model

Another key class for the image captioning model is the `TransformerEncoderBlock` shown in the cell below. Execute this code to load this class.   

In [ ]:
class TransformerEncoderBlock(layers.Layer):
    def __init__(self, embed_dim, dense_dim, num_heads, **kwargs):
        super().__init__(**kwargs)
        self.embed_dim = embed_dim
        self.dense_dim = dense_dim
        self.num_heads = num_heads
        self.attention_1 = layers.MultiHeadAttention(
            num_heads=num_heads, key_dim=embed_dim, dropout=0.0
        )
        self.layernorm_1 = layers.LayerNormalization()
        self.layernorm_2 = layers.LayerNormalization()
        self.dense_1 = layers.Dense(embed_dim, activation="relu")

    def call(self, inputs, training, mask=None):
        inputs = self.layernorm_1(inputs)
        inputs = self.dense_1(inputs)

        attention_output_1 = self.attention_1(
            query=inputs,
            value=inputs,
            key=inputs,
            attention_mask=None,
            training=training,
        )
        out_1 = self.layernorm_2(inputs + attention_output_1)
        return out_1

> **Exercise 6-1:** Does this transformer block use self attention and how do you know?

> **Answer:**    

The `PositionalEmbedding` object adds positional encoding to the input token sequence. Execute this code to load this class.

In [ ]:
class PositionalEmbedding(layers.Layer):
    def __init__(self, sequence_length, vocab_size, embed_dim, **kwargs):
        super().__init__(**kwargs)
        self.token_embeddings = layers.Embedding(
            input_dim=vocab_size, output_dim=embed_dim
        )
        self.position_embeddings = layers.Embedding(
            input_dim=sequence_length, output_dim=embed_dim
        )
        self.sequence_length = sequence_length
        self.vocab_size = vocab_size
        self.embed_dim = embed_dim
        self.embed_scale = tf.math.sqrt(tf.cast(embed_dim, tf.float32))

    def call(self, inputs):
        length = tf.shape(inputs)[-1]
        positions = tf.range(start=0, limit=length, delta=1)
        embedded_tokens = self.token_embeddings(inputs)
        embedded_tokens = embedded_tokens * self.embed_scale
        embedded_positions = self.position_embeddings(positions)
        return embedded_tokens + embedded_positions

    def compute_mask(self, inputs, mask=None):
        return tf.math.not_equal(inputs, 0)

> **Exercise 6-2:** Examine the code in the `call` method above and answer these questions.
> 1. How are the positions of the tokens encoded?
> 2. Are the position embeddings learned and why?
> 3. How are the embeddings integrated with the tokens?

> **Answers:**
> 1.      
> 2.      
> 3.     

The code in the cell below defines the `TransformerDecoderBlock` class with the following methods.       
1. The **call** method applies masked self-attention to the text input.     
2. The **`get_causal_attention_mask`** generates the causal attention mask applied to the attention calculations.

Execute this code to load the code for this class.  

In [ ]:
class TransformerDecoderBlock(layers.Layer):
    def __init__(self, embed_dim, ff_dim, num_heads, **kwargs):
        super().__init__(**kwargs)
        self.embed_dim = embed_dim
        self.ff_dim = ff_dim
        self.num_heads = num_heads
        self.attention_1 = layers.MultiHeadAttention(
            num_heads=num_heads, key_dim=embed_dim, dropout=0.1
        )
        self.attention_2 = layers.MultiHeadAttention(
            num_heads=num_heads, key_dim=embed_dim, dropout=0.1
        )
        self.ffn_layer_1 = layers.Dense(ff_dim, activation="relu")
        self.ffn_layer_2 = layers.Dense(embed_dim)

        self.layernorm_1 = layers.LayerNormalization()
        self.layernorm_2 = layers.LayerNormalization()
        self.layernorm_3 = layers.LayerNormalization()

        self.embedding = PositionalEmbedding(
            embed_dim=EMBED_DIM,
            sequence_length=SEQ_LENGTH,
            vocab_size=VOCAB_SIZE,
        )
        self.out = layers.Dense(VOCAB_SIZE, activation="softmax")

        self.dropout_1 = layers.Dropout(0.3)
        self.dropout_2 = layers.Dropout(0.5)
        self.supports_masking = True

    def call(self, inputs, encoder_outputs, training, mask=None):
        inputs = self.embedding(inputs)
        causal_mask = self.get_causal_attention_mask(inputs)

        if mask is not None:
            padding_mask = tf.cast(mask[:, :, tf.newaxis], dtype=tf.int32)
            combined_mask = tf.cast(mask[:, tf.newaxis, :], dtype=tf.int32)
            combined_mask = tf.minimum(combined_mask, causal_mask)

        attention_output_1 = self.attention_1(
            query=inputs,
            value=inputs,
            key=inputs,
            attention_mask=combined_mask,
            training=training,
        )
        out_1 = self.layernorm_1(inputs + attention_output_1)

        attention_output_2 = self.attention_2(
            query=out_1,
            value=encoder_outputs,
            key=encoder_outputs,
            attention_mask=padding_mask,
            training=training,
        )
        out_2 = self.layernorm_2(out_1 + attention_output_2)

        ffn_out = self.ffn_layer_1(out_2)
        ffn_out = self.dropout_1(ffn_out, training=training)
        ffn_out = self.ffn_layer_2(ffn_out)

        ffn_out = self.layernorm_3(ffn_out + out_2, training=training)
        ffn_out = self.dropout_2(ffn_out, training=training)
        preds = self.out(ffn_out)
        return preds

    def get_causal_attention_mask(self, inputs):
        input_shape = tf.shape(inputs)
        batch_size, sequence_length = input_shape[0], input_shape[1]
        i = tf.range(sequence_length)[:, tf.newaxis]
        j = tf.range(sequence_length)
        mask = tf.cast(i >= j, dtype="int32")
        mask = tf.reshape(mask, (1, input_shape[1], input_shape[1]))
        mult = tf.concat(
            [
                tf.expand_dims(batch_size, -1),
                tf.constant([1, 1], dtype=tf.int32),
            ],
            axis=0,
        )
        return tf.tile(mask, mult)

> **Exercise 6-3:** Explain why masking is required for the attention calculations in the TransformerDecoderBlock class.   

> **Answer:**       


To pull the foregoing together we need code to train and test the model. The `ImageCaptioningModel` object performs these functions using the following methods.     
1. The **`calculate_loss`** method computes the loss when training the model.
2. The **`calculate_accuracy`** method computes the accuracy of the model.
3. The **`_compute_caption_loss_and_acc`** method is a helper function which calls the decoder and computes the loss and accuracy.
4. The **`train_step`** method executes a gradient descent step leaning step using the foregoing methods.
5. The **`test_step`** method computes the loss and accuracy of inferences by the model image by image.

Execute this code to instantiate these objects.

In [ ]:
class ImageCaptioningModel(keras.Model):
    def __init__(
        self,
        cnn_model,
        encoder,
        decoder,
        num_captions_per_image=5,
        image_aug=None,
    ):
        super().__init__()
        self.cnn_model = cnn_model
        self.encoder = encoder
        self.decoder = decoder
        self.loss_tracker = keras.metrics.Mean(name="loss")
        self.acc_tracker = keras.metrics.Mean(name="accuracy")
        self.num_captions_per_image = num_captions_per_image
        self.image_aug = image_aug

    def calculate_loss(self, y_true, y_pred, mask):
        loss = self.loss(y_true, y_pred)
        mask = tf.cast(mask, dtype=loss.dtype)
        loss *= mask
        return tf.reduce_sum(loss) / tf.reduce_sum(mask)

    def calculate_accuracy(self, y_true, y_pred, mask):
        accuracy = tf.equal(y_true, tf.argmax(y_pred, axis=2))
        accuracy = tf.math.logical_and(mask, accuracy)
        accuracy = tf.cast(accuracy, dtype=tf.float32)
        mask = tf.cast(mask, dtype=tf.float32)
        return tf.reduce_sum(accuracy) / tf.reduce_sum(mask)

    def _compute_caption_loss_and_acc(self, img_embed, batch_seq, training=True):
        encoder_out = self.encoder(img_embed, training=training)
        batch_seq_inp = batch_seq[:, :-1]
        batch_seq_true = batch_seq[:, 1:]
        mask = tf.math.not_equal(batch_seq_true, 0)
        batch_seq_pred = self.decoder(
            batch_seq_inp, encoder_out, training=training, mask=mask
        )
        loss = self.calculate_loss(batch_seq_true, batch_seq_pred, mask)
        acc = self.calculate_accuracy(batch_seq_true, batch_seq_pred, mask)
        return loss, acc

    def train_step(self, batch_data):
        batch_img, batch_seq = batch_data
        batch_loss = 0
        batch_acc = 0

        if self.image_aug:
            batch_img = self.image_aug(batch_img)

        # Get image embeddings
        img_embed = self.cnn_model(batch_img)

        # Pass each of the five captions one by one to the decoder
        # along with the encoder outputs and compute the loss as well as accuracy
        # for each caption.
        for i in range(self.num_captions_per_image):
            with tf.GradientTape() as tape:
                loss, acc = self._compute_caption_loss_and_acc(
                    img_embed, batch_seq[:, i, :], training=True
                )

                # Update loss and accuracy
                batch_loss += loss
                batch_acc += acc

            # Get the list of all the trainable weights
            train_vars = (
                self.encoder.trainable_variables + self.decoder.trainable_variables
            )

            # Get the gradients
            grads = tape.gradient(loss, train_vars)

            # Update the trainable weights
            self.optimizer.apply_gradients(zip(grads, train_vars))

        # 7. Update the trackers
        batch_acc /= float(self.num_captions_per_image)
        self.loss_tracker.update_state(batch_loss)
        self.acc_tracker.update_state(batch_acc)

        # 8. Return the loss and accuracy values
        return {
            "loss": self.loss_tracker.result(),
            "acc": self.acc_tracker.result(),
        }

    def test_step(self, batch_data):
        batch_img, batch_seq = batch_data
        batch_loss = 0
        batch_acc = 0

        # 1. Get image embeddings
        img_embed = self.cnn_model(batch_img)

        # 2. Pass each of the five captions one by one to the decoder
        # along with the encoder outputs and compute the loss as well as accuracy
        # for each caption.
        for i in range(self.num_captions_per_image):
            loss, acc = self._compute_caption_loss_and_acc(
                img_embed, batch_seq[:, i, :], training=False
            )

            # 3. Update batch loss and batch accuracy
            batch_loss += loss
            batch_acc += acc

        batch_acc /= float(self.num_captions_per_image)

        # 4. Update the trackers
        self.loss_tracker.update_state(batch_loss)
        self.acc_tracker.update_state(batch_acc)

        # 5. Return the loss and accuracy values
        return {
            "loss": self.loss_tracker.result(),
            "acc": self.acc_tracker.result(),
        }

    @property
    def metrics(self):
        # We need to list our metrics here so the `reset_states()` can be
        # called automatically.
        return [self.loss_tracker, self.acc_tracker]

> **Exercise 6-4:**
> 1. In brief, explain why masking is required to compute the loss.    
> 2. In the `train_step` method, the `_compute_caption_loss_and_acc` calls the `encoder` and then the `decoder`. The encoder operates on the image embedding. Examine the arguments to the call to the `decoder`, as well as the code for this method. Explain what the `attention_2` block is doing and why this is essential for the multi-modal model to operate. Pay attention to the query, key and value arguments.   
> 3. In brief, explain the four major steps of the learning process in the `train_step` method.    
> 4. VAEs have a latent embedding space. For this multi-model model describe the latent space and state its dimensionality.    

> **Answers:**`
> 1.     
> 2.      
> 3.     
>    a.      
>    b.      
>    c.     
>    d.     
> 4.     

Finally, we are ready to instantiate the components of our model by executing the code in the cell below.    

In [ ]:
cnn_model = get_cnn_model()
encoder = TransformerEncoderBlock(embed_dim=EMBED_DIM, dense_dim=FF_DIM, num_heads=1)
decoder = TransformerDecoderBlock(embed_dim=EMBED_DIM, ff_dim=FF_DIM, num_heads=2)
caption_model = ImageCaptioningModel(
    cnn_model=cnn_model,
    encoder=encoder,
    decoder=decoder,
    image_aug=image_augmentation,
)

## Model training

Execute the code in the cell below to train the model which should take approximately 20-30 min.

In [ ]:
post_warmup_learning_rate=1e-5
weight_decay=0.0001
warmup_steps=num_warmup_steps_divisor=15

# Define the loss function
cross_entropy = keras.losses.SparseCategoricalCrossentropy(
    from_logits=False,
    reduction=None,
)

# EarlyStopping criteria
early_stopping = keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True)


# Learning Rate Scheduler for the optimizer
class LRSchedule(keras.optimizers.schedules.LearningRateSchedule):
    def __init__(self, post_warmup_learning_rate, warmup_steps):
        super().__init__()
        self.post_warmup_learning_rate = post_warmup_learning_rate
        self.warmup_steps = warmup_steps

    def __call__(self, step):
        global_step = tf.cast(step, tf.float32)
        warmup_steps = tf.cast(self.warmup_steps, tf.float32)
        warmup_progress = global_step / warmup_steps
        warmup_learning_rate = self.post_warmup_learning_rate * warmup_progress
        return tf.cond(
            global_step < warmup_steps,
            lambda: warmup_learning_rate,
            lambda: self.post_warmup_learning_rate,
        )


# Create a learning rate schedule
num_train_steps = len(train_dataset) * EPOCHS
num_warmup_steps = num_train_steps // num_warmup_steps_divisor
lr_schedule = LRSchedule(post_warmup_learning_rate=post_warmup_learning_rate,
                         warmup_steps=num_warmup_steps)

# Compile the model
caption_model.compile(optimizer=keras.optimizers.Adam(lr_schedule,
                                                      weight_decay=weight_decay),
                      loss=cross_entropy,
                      metrics=['accuracy'])


# Fit the model
history = caption_model.fit(
    train_dataset,
    epochs=EPOCHS,
    validation_data=valid_dataset,
    callbacks=[early_stopping],
)

In [ ]:
def plot_hist(hist):
    _,ax = plt.subplots(1,2, figsize = (12,6))
    ax[0].plot(hist.history["acc"], label="train")
    ax[0].plot(hist.history["val_acc"], label="validation")
    ax[0].set_title("model accuracy")
    ax[0].set_ylabel("accuracy")
    ax[0].set_xlabel("epoch")
    ax[0].legend(loc="upper left")
    ax[1].plot(hist.history["loss"], label="train")
    ax[1].plot(hist.history["val_loss"], label="validation")
    ax[1].set_title("model loss")
    ax[1].set_ylabel("loss")
    ax[1].set_xlabel("epoch")
    ax[1].legend(loc="upper right")
    plt.show()

plot_hist(history)

> **Exercise 6-5**:
> 1. The [keras.losses.SparseCategoricalCrossentropy](keras.losses.SparseCategoricalCrossentropy) function is used as the loss. Carefully examine the documentation, noticing that `from_logits=False` argument is for "By default, we assume that y_pred encodes a probability distribution." Why is this a function the correct choice for the loss between the generated and label captions?
> 2. Explain how the the categorical cross entropy function fits with the Beta-VAE loss, including the trade-off parameter.                 
> 3. Examine the `calculate_loss` in the `ImageCaptioningModel` class. Explain how the loss is stabilized over the batch.             
> 4. Given the code to compute accuracy you have already reviewed, how can you interpret the values of the training and validation accuracy displayed during the training epochs?      
> 5. Given the learning curves, does it appear that the model exhibits significant over-fitting and why?     

> **Answers:**
> 1.      
> 2.      
> 4.        
> 5.       

## Check sample predictions

Now for the fun part! Execute the code in the cell below to generate some captions for three randomly sampled validation images.You can run this code several times to see multiple examples.     

We are not performing additional quantitative evaluation in the interest of brevity. For a production solution additional quantitative evaluation is required applying metrics familiar from NLP modeling.    

In [ ]:
vocab = vectorization.get_vocabulary()
index_lookup = dict(zip(range(len(vocab)), vocab))
max_decoded_sentence_length = SEQ_LENGTH - 1
valid_images = list(valid_data.keys())


def generate_caption():
    # Select a random image from the validation dataset
    sample_img = np.random.choice(valid_images)

    # Read the image from the disk
    sample_img = decode_and_resize(sample_img)
    img = sample_img.numpy().clip(0, 255).astype(np.uint8)
    plt.imshow(img)
    plt.show()

    # Pass the image to the CNN
    img = tf.expand_dims(sample_img, 0)
    img = caption_model.cnn_model(img)

    # Pass the image features to the Transformer encoder
    encoded_img = caption_model.encoder(img, training=False)

    # Generate the caption using the Transformer decoder
    decoded_caption = "<start> "
    for i in range(max_decoded_sentence_length):
        tokenized_caption = vectorization([decoded_caption])[:, :-1]
        mask = tf.math.not_equal(tokenized_caption, 0)
        predictions = caption_model.decoder(
            tokenized_caption, encoded_img, training=False, mask=mask
        )
        sampled_token_index = np.argmax(predictions[0, i, :])
        sampled_token = index_lookup[sampled_token_index]
        if sampled_token == "<end>":
            break
        decoded_caption += " " + sampled_token

    decoded_caption = decoded_caption.replace("<start> ", "")
    decoded_caption = decoded_caption.replace(" <end>", "").strip()
    print("Predicted Caption: ", decoded_caption)


# Check predictions for a few samples
generate_caption()
generate_caption()
generate_caption()

#### Portions of this document are copyright 2026, Stephen F Elston. All rights reserved.  